# ShowDiffraction demo

In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


## 1 - Polycrystalline 

In [2]:
import numpy as np
import torch

dev = torch.device("mps" if torch.backends.mps.is_available()
                   else "cuda" if torch.cuda.is_available() else "cpu")

#debye-scherrer rings
N = 512
CY, CX = N / 2 + 9, N / 2 - 6           # the TRUE ring/beam center
RINGS_PX = [58.0, 95.0, 138.0, 176.0, 205.0]
yy, xx = torch.meshgrid(torch.arange(N), torch.arange(N), indexing="ij")
r = torch.sqrt((yy.to(dev) - CY) ** 2 + (xx.to(dev) - CX) ** 2)
dp_poly = 90.0 * torch.exp(-(r ** 2) / (2 * 6.0 ** 2))            # transmitted beam
for radius_px, amp in zip(RINGS_PX, [9.0, 6.0, 7.0, 3.5, 4.0]):
    dp_poly += amp * torch.exp(-((r - radius_px) ** 2) / (2 * 2.2 ** 2))
dp_poly += 0.05 * torch.rand(N, N, device=dev)
dp_poly = dp_poly.cpu().numpy().astype(np.float32)
print("polycrystalline DP:", dp_poly.shape)

polycrystalline DP: (512, 512)


In [3]:
from quantem.widget import ShowDiffraction

BAD_CENTER = (CY - 90, CX - 88)    #offcenter
w_poly = ShowDiffraction(
    dp_poly,
    title="Polycrystalline SAED",
    center=BAD_CENTER,
    show_radial=True,
)
print(f"start center (incorrect) = ({w_poly.center_row:.0f}, {w_poly.center_col:.0f})   "
      f"(true) ring center ≈ ({CY:.0f}, {CX:.0f})")
w_poly

  to cpu: 0.05s (1.0 MB)
start center (incorrect) = (175, 162)   (true) ring center ≈ (265, 250)


/workspaces/_quantem-workspace/quantem.widget/.venv/lib/python3.14/site-packages/anywidget/_util.py:283: UserWarning: anywidget: Live-reloading feature is disabled. To enable, please install the 'watchfiles' package.
  start_thread=_should_start_thread(path),


ShowDiffraction(shape=(1, 1, 512, 512), sampling=(1.0 Å, 0.0 px), pos=(0, 0), title='Polycrystalline SAED')

The white center crosshair starts in the wrong place, and the I(q) peaks are smeared/incorrect because the azimuthal average is taken around the wrong point

Steps:
1. Center: Ring (3pt), then click 3 well-spread points on any one ring
2. On the 3rd click the crosshair jumps to the ring center and the I(q) should snap to some sharp peaks
3. Click each sharp peak in the I(q) plot to add to ring table
4. Type `2.34` in Calibrate d (Å) and click FROM RING, so then the ring spacings fill in

## 2 - Single crystal

In [4]:
# single-crystal SAED
G = 42.0        # reciprocal lattice spacing
hk = np.arange(-5, 6)
centers, amps = [], []
for h in hk:
    for k in hk:
        rr, cc = CY + h * G, CX + k * G
        if 0 <= rr < N and 0 <= cc < N:
            centers.append((rr, cc))
            g = np.hypot(h, k)
            amps.append(120.0 if g == 0 else 9.0 / (1.0 + g))  # beam brightest
centers = np.array(centers, dtype=np.float32)
sr = torch.tensor(centers[:, 0], device=dev).view(-1, 1, 1)
sc = torch.tensor(centers[:, 1], device=dev).view(-1, 1, 1)
amp = torch.tensor(amps, device=dev).view(-1, 1, 1)
r2 = (yy.to(dev) - sr) ** 2 + (xx.to(dev) - sc) ** 2
dp_sc = (amp * torch.exp(-r2 / (2 * 2.2 ** 2))).sum(0)
dp_sc += 0.05 * torch.rand(N, N, device=dev)
dp_sc = dp_sc.cpu().numpy().astype(np.float32)
print("single-crystal DP:", dp_sc.shape)

single-crystal DP: (512, 512)


In [5]:
BAD_CENTER_SC = (CY + 85, CX + 80)     # ~117 px off
w_sc = ShowDiffraction(
    dp_sc,
    title="Single-crystal SAED",
    center=BAD_CENTER_SC,
    snap_enabled=True,
)
print(f"start center (wrong) = ({w_sc.center_row:.0f}, {w_sc.center_col:.0f})   "
      f"true beam center ≈ ({CY:.0f}, {CX:.0f})")
w_sc

  to cpu: 0.03s (1.0 MB)
start center (wrong) = (350, 330)   true beam center ≈ (265, 250)


ShowDiffraction(shape=(1, 1, 512, 512), sampling=(1.0 Å, 0.0 px), pos=(0, 0), title='Single-crystal SAED')

The center crosshair starts off the bright central beam 

1. Center: Midpoint, then click a bright spot and a spot directly opposite it across the pattern (Friedel pair g and -g).
2. On the 2nd click the crosshair jumps to the midpoint, it should land right on the central beam, with the spot pattern symmetric about it
3. With Snap on, click a few spots, and the Spots table will fill with positions.

## 3 - real data 

In [ ]:
import glob
import numpy as np
from quantem.widget import IO, ShowDiffraction

DM4_PATH = ""
if not DM4_PATH:
    hits = sorted(glob.glob("/workspaces/_quantem-workspace/diffraction_reference/*.dm4"))
    DM4_PATH = hits[0] if hits else ""
assert DM4_PATH, "Set DM4_PATH to a .dm4 file."

res = IO.file(DM4_PATH)
dp = np.asarray(res.data, dtype=np.float32)

bin_factor = max(1, min(dp.shape) // 1024)
if bin_factor > 1:
    h = (dp.shape[0] // bin_factor) * bin_factor
    w = (dp.shape[1] // bin_factor) * bin_factor
    dp = dp[:h, :w].reshape(h // bin_factor, bin_factor, w // bin_factor, bin_factor).mean((1, 3))

k_per_A = (res.pixel_size * 0.1 * bin_factor) if (res.units and "nm" in str(res.units)) else 0.0
print(f"{res.title}  |  {res.data.shape} → display {dp.shape} (bin {bin_factor}×)  |  "
      f"k = {k_per_A:.5f} 1/Å/px")

w_real = ShowDiffraction(dp.astype(np.float32), title=res.title, k_pixel_size=k_per_A, show_radial=True)
w_real

IO.file: 4096×4096 64 MB in 441 ms  pixel_size=0.0091 Å
20251027-JC7-A7-10%Ni-12%Ce-HSA-post rxn_0038  |  (4096, 4096) → display (1024, 1024) (bin 4×)  |  k = 0.00364 1/Å/px
  to cpu: 0.09s (4.2 MB)


ShowDiffraction(shape=(1, 1, 1024, 1024), sampling=(1.0 Å, 0.003640929237008095 1/Å), pos=(0, 0), title='20251027-JC7-A7-10%Ni-12%Ce-HSA-post rxn_0038')